# AlloHub v2 AI 모델 테스트 및 수치 검증 노트북

이 노트북은 **AlloHub v2**의 핵심 AI 모델 및 알고리즘을 종합적으로 테스트하고 실제 수치 계산을 수행합니다.

### 검증 대상 및 구성:
1. **데이터 적재 및 수치 통계 분석**: 공공데이터 6종 기반 상장종목(2,737개사)의 시장 분포, 위험 라벨(부도·영업정지·회생), 배당, 감자, 순이익 통계
2. **피처 엔지니어링 & 전처리 검증**: 범주형 원-핫 인코딩, 수치형 스케일링, 감자 상한 클리핑(최대 10건) 검증
3. **머신러닝(RandomForest) 모델 수치 평가**: `risk_rf.joblib` 성능 지표 (Accuracy, Precision, Recall, F1, ROC-AUC), 혼동 행렬, 피처 중요도 분석
4. **딥러닝(PyTorch MLP) 모델 수치 평가**: `risk_mlp.pt` 신경망 성능 지표, 손실값, ML vs DL 수치 비교 및 상관관계 분석
5. **앙상블 종합 점수 및 등급 규칙 수치 검증**: ML+DL 평균 점수 분포, 위험 등급(안정·주의·위험) 분포 및 공시 사실 우선 규칙 검증
6. **기능 2 - 출자금 원 단위 배분(Allocation Split) 수치 계산**: 백엔드 AllocationCalculator와 일치하는 원 단위 분할 및 절사 잔여액 배분 검증
7. **실제 상장 종목 E2E 진단 케이스 스터디**: 삼성전자(KOSPI 대형주), 카카오, 감자 발생 종목, 고위험 공시 종목 등 실제 진단 수치 확인

In [1]:
import sys
import json
from pathlib import Path
from decimal import Decimal, ROUND_DOWN
import pandas as pd
import numpy as np
import torch

# 프로젝트 루트 및 ai 경로 동적 감지
curr = Path.cwd().resolve()
if (curr / 'ai').exists():
    PROJECT_ROOT = curr
elif (curr.parent / 'ai').exists():
    PROJECT_ROOT = curr.parent
else:
    PROJECT_ROOT = Path('..').resolve()

AI_DIR = PROJECT_ROOT / 'ai'
if str(AI_DIR) not in sys.path:
    sys.path.insert(0, str(AI_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 모듈화된 패키지 임포트
from src.core.config import (
    MODELS_DIR, DATA_CSV, MODELS_CSV,
    MODEL_RF_PATH, MODEL_MLP_PATH, PREPROCESSOR_MLP_PATH,
    GRADE_NAMES, SCORE_THRESHOLD_WARNING
)
from src.scoring.features import (
    FEATURE_COLUMNS, CATEGORY_COLUMNS, NUMBER_COLUMNS,
    TARGET_COLUMN, make_input_frame
)
from src.scoring.rules import grade_of
from src.scoring.company import load_companies, find_company, normalize_name
from src.inference.ml_risk import load_ml_model, predict_ml
from src.inference.dl_risk import load_dl_model, predict_dl, RiskMLP
from src.service.screener import screen

print('✓ 프로젝트 환경 설정 완료')
print(f'  - Project Root: {PROJECT_ROOT}')
print(f'  - AI Directory: {AI_DIR}')
print(f'  - PyTorch Version: {torch.__version__}')

✓ 프로젝트 환경 설정 완료
  - Project Root: /Users/jeong-yelim/.gemini/antigravity/scratch/AlloHub_v2
  - AI Directory: /Users/jeong-yelim/.gemini/antigravity/scratch/AlloHub_v2/ai
  - PyTorch Version: 2.14.1


## 1. 데이터 적재 및 수치 통계 분석
공공데이터 API 6종(KRX상장정보, 배당공시, 부도, 영업정지, 회생, 감자)을 전처리한 전체 상장종목 데이터셋을 로드하고 주요 수치 통계를 확인합니다.

In [2]:
df = load_companies()
print(f'총 상장 기업 수: {len(df):,}개사')
print(f'데이터셋 컬럼 수: {df.shape[1]}개')
print('\n[상위 5개사 데이터 미리보기]')
display_cols = ['item_name', 'market', 'has_dividend', 'dividend_rate', 'profit_negative', 'net_profit', 'reduction_count', 'distress']
df[display_cols].head()

총 상장 기업 수: 2,737개사
데이터셋 컬럼 수: 17개

[상위 5개사 데이터 미리보기]
Out[0]: 
  item_name  market  has_dividend  ...  net_profit  reduction_count  distress
0  오가닉티코스메틱  KOSDAQ             0  ...         0.0                0         0
1      동화약품   KOSPI             1  ...     28237.0                0         0
2     KR모터스   KOSPI             1  ...    -21027.0                3         1
3        경방   KOSPI             1  ...    -13452.0                0         0
4     삼양홀딩스   KOSPI             1  ...    172366.0                0         0

[5 rows x 8 columns]


,item_name,market,has_dividend,dividend_rate,profit_negative,net_profit,reduction_count,distress
0,오가닉티코스메틱,KOSDAQ,0,0.00,0,0.0,0,0
1,동화약품,KOSPI,1,2.83,0,28237.0,0,0
2,KR모터스,KOSPI,1,0.00,1,-21027.0,3,1
3,경방,KOSPI,1,0.00,1,-13452.0,0,0
4,삼양홀딩스,KOSPI,1,5.20,0,172366.0,0,0


In [3]:
# 위험 라벨 및 공시 통계
distress_counts = df['distress'].value_counts()
distress_rate = df['distress'].mean() * 100

default_cos = int((df['default_count'] > 0).sum())
suspend_cos = int((df['suspend_count'] > 0).sum())
rehab_cos = int((df['rehab_count'] > 0).sum())
reduction_cos = int((df['reduction_count'] > 0).sum())

stats_summary = pd.DataFrame({
    '구분': ['정상 기업 (distress=0)', '위험 기업 (distress=1)', '부도 공시 기업', '영업정지 공시 기업', '회생 공시 기업', '감자 공시 기업'],
    '기업 수': [int(distress_counts.get(0, 0)), int(distress_counts.get(1, 0)), default_cos, suspend_cos, rehab_cos, reduction_cos],
    '비율 (%)': [
        round(100 - distress_rate, 2),
        round(distress_rate, 2),
        round(default_cos / len(df) * 100, 2),
        round(suspend_cos / len(df) * 100, 2),
        round(rehab_cos / len(df) * 100, 2),
        round(reduction_cos / len(df) * 100, 2)
    ]
})
print('=== 상장 기업 위험 및 공시 현황 ===')
stats_summary

=== 상장 기업 위험 및 공시 현황 ===
Out[0]: 
                   구분  기업 수  비율 (%)
0  정상 기업 (distress=0)  2504   91.49
1  위험 기업 (distress=1)   233    8.51
2            부도 공시 기업     8    0.29
3          영업정지 공시 기업   172    6.28
4            회생 공시 기업    71    2.59
5            감자 공시 기업   337   12.31


,구분,기업 수,비율 (%)
0,정상 기업 (distress=0),2504,91.49
1,위험 기업 (distress=1),233,8.51
2,부도 공시 기업,8,0.29
3,영업정지 공시 기업,172,6.28
4,회생 공시 기업,71,2.59
5,감자 공시 기업,337,12.31


In [4]:
# 시장별 기업 수 및 위험 비율
market_summary = df.groupby('market').agg(
    총기업수=('item_name', 'count'),
    위험기업수=('distress', 'sum'),
    위험비율_pct=('distress', lambda x: round(x.mean() * 100, 2)),
    평균배당률=('dividend_rate', lambda x: round(x.mean(), 2)),
    적자기업비율_pct=('profit_negative', lambda x: round(x.mean() * 100, 2))
).reset_index()

print('=== 시장별 기업 분포 및 위험 지표 ===')
market_summary

=== 시장별 기업 분포 및 위험 지표 ===
Out[0]: 
   market  총기업수  위험기업수  위험비율_pct  평균배당률  적자기업비율_pct
0   KONEX   107     11     10.28   0.04       21.50
1  KOSDAQ  1801    136      7.55   0.81       32.37
2   KOSPI   829     86     10.37   1.96       23.28


,market,총기업수,위험기업수,위험비율_pct,평균배당률,적자기업비율_pct
0,KONEX,107,11,10.28,0.04,21.50
1,KOSDAQ,1801,136,7.55,0.81,32.37
2,KOSPI,829,86,10.37,1.96,23.28


In [5]:
# 주요 수치형 변수 기술통계량
num_cols = ['dividend_rate', 'net_profit', 'log_profit', 'par_price', 'reduction_count']
desc = df[num_cols].describe().round(2)
print('=== 주요 수치형 변수 요약 통계 ===')
print(desc)

# 타겟(distress)과의 상관계수 계산
correlations = {}
for col in ['has_dividend', 'dividend_rate', 'profit_negative', 'log_profit', 'par_price', 'reduction_count']:
    correlations[col] = {
        '피어슨 상관계수': round(float(df[col].corr(df['distress'])), 4),
        '스피어만 순위상관계수': round(float(df[col].corr(df['distress'], method='spearman')), 4)
    }
corr_df = pd.DataFrame(correlations).T
print('\n=== 피처와 위험 라벨(distress) 간 상관관계 ===')
corr_df

=== 주요 수치형 변수 요약 통계 ===
       dividend_rate    net_profit  log_profit  par_price  reduction_count
count        2737.00  2.737000e+03     2737.00    2737.00          2737.00
mean            1.13 -1.006781e+07        7.68     902.84             0.33
std             2.84  3.787812e+08        4.02    1360.08             1.26
min             0.00 -1.531914e+10        0.00       0.00             0.00
25%             0.00 -2.327000e+03        7.19     500.00             0.00
50%             0.00  6.810000e+02        8.92     500.00             0.00
75%             1.61  1.384500e+04       10.12     500.00             0.00
max           100.00  2.258981e+08       23.45   10000.00            14.00

=== 피처와 위험 라벨(distress) 간 상관관계 ===
Out[0]: 
                 피어슨 상관계수  스피어만 순위상관계수
has_dividend       0.0352       0.0352
dividend_rate     -0.0376      -0.0685
profit_negative    0.0950       0.0950
log_profit         0.0150       0.0059
par_price          0.0398       0.0676
reduction_count    0.3

,피어슨 상관계수,스피어만 순위상관계수
has_dividend,0.0352,0.0352
dividend_rate,-0.0376,-0.0685
profit_negative,0.0950,0.0950
log_profit,0.0150,0.0059
par_price,0.0398,0.0676
reduction_count,0.3599,0.3165


## 2. 피처 엔지니어링 및 전처리 수치 검증
모델 입력으로 사용되는 7개 피처(`market`, `has_dividend`, `dividend_rate`, `profit_negative`, `log_profit`, `par_price`, `reduction_count`)와 감자 건수 클리핑 규칙(최대 10건)을 검증합니다.

In [6]:
# 임의의 테스트 샘플로 make_input_frame 동작 검증
sample_row = df.iloc[0].to_dict()
sample_input = make_input_frame(sample_row)

print('피처 컬럼 목록:', FEATURE_COLUMNS)
print('생성된 피처 프레임 형태:', sample_input.shape)
print('\n피처 프레임 내용:')
print(sample_input.to_dict(orient='records')[0])

# 감자 상한(10) 클리핑 테스트
test_high_reduction = sample_row.copy()
test_high_reduction['reduction_count'] = 45
clipped_input = make_input_frame(test_high_reduction)
print(f'\n감자 45건 입력 시 클리핑 결과: {clipped_input["reduction_count"].iloc[0]}건 (기대값: 10)')
assert clipped_input['reduction_count'].iloc[0] == 10, '감자 클리핑 오류'
print('✓ 피처 전처리 및 클리핑 검증 통과')

피처 컬럼 목록: ['market', 'has_dividend', 'dividend_rate', 'profit_negative', 'log_profit', 'par_price', 'reduction_count']
생성된 피처 프레임 형태: (1, 7)

피처 프레임 내용:
{'market': 'KOSDAQ', 'has_dividend': 0, 'dividend_rate': 0.0, 'profit_negative': 0, 'log_profit': 0.0, 'par_price': 0.0, 'reduction_count': 0}

감자 45건 입력 시 클리핑 결과: 10건 (기대값: 10)
✓ 피처 전처리 및 클리핑 검증 통과


## 3. 머신러닝(RandomForest) 모델 수치 평가
`models/risk_rf.joblib` 모델의 정확도, F1-Score, ROC-AUC, 혼동 행렬 및 피처 중요도(Feature Importance)를 산출합니다.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

ml_model = load_ml_model()

# 동일한 재현성을 갖는 학습/테스트 분할 (80:20 Stratified)
X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 예측 수행
y_prob_ml = ml_model.predict_proba(X_test)[:, 1]
y_pred_ml = (y_prob_ml >= 0.5).astype(int)

# 지표 계산
acc_ml = float(accuracy_score(y_test, y_pred_ml))
prec_ml = float(precision_score(y_test, y_pred_ml, zero_division=0))
rec_ml = float(recall_score(y_test, y_pred_ml, zero_division=0))
f1_ml = float(f1_score(y_test, y_pred_ml, zero_division=0))
auc_ml = float(roc_auc_score(y_test, y_prob_ml))

cm_ml = confusion_matrix(y_test, y_pred_ml)

print('=== 머신러닝(RandomForest) 평가 결과 (테스트셋 N=548) ===')
print(f'Accuracy  : {acc_ml:.4f} ({acc_ml*100:.2f}%)')
print(f'Precision : {prec_ml:.4f}')
print(f'Recall    : {rec_ml:.4f}')
print(f'F1-Score  : {f1_ml:.4f}')
print(f'ROC-AUC   : {auc_ml:.4f}')

cm_df_ml = pd.DataFrame(
    cm_ml,
    index=['실제 정상(0)', '실제 위험(1)'],
    columns=['예측 정상(0)', '예측 위험(1)']
)
print('\n[혼동 행렬 (Confusion Matrix)]')
cm_df_ml

=== 머신러닝(RandomForest) 평가 결과 (테스트셋 N=548) ===
Accuracy  : 0.8394 (83.94%)
Precision : 0.2469
Recall    : 0.4255
F1-Score  : 0.3125
ROC-AUC   : 0.6979

[혼동 행렬 (Confusion Matrix)]
Out[0]: 
          예측 정상(0)  예측 위험(1)
실제 정상(0)       440        61
실제 위험(1)        27        20


,예측 정상(0),예측 위험(1)
실제 정상(0),440,61
실제 위험(1),27,20


In [8]:
# 파이프라인 내부의 원-핫 인코더 피처 이름과 피처 중요도 추출
rf_estimator = ml_model.named_steps['model']
preprocessor = ml_model.named_steps['preprocess']

cat_features = preprocessor.named_transformers_['category'].get_feature_names_out(CATEGORY_COLUMNS).tolist()
all_feature_names = cat_features + NUMBER_COLUMNS
importances = rf_estimator.feature_importances_

fi_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances,
    'Percentage (%)': (importances / importances.sum() * 100).round(2)
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('=== 머신러닝 피처 중요도 순위 (Feature Importance Ranking) ===')
fi_df

=== 머신러닝 피처 중요도 순위 (Feature Importance Ranking) ===
Out[0]: 
           Feature  Importance  Percentage (%)
0  reduction_count    0.437363           43.74
1       log_profit    0.240183           24.02
2    dividend_rate    0.127275           12.73
3        par_price    0.091144            9.11
4  profit_negative    0.059333            5.93
5     market_KOSPI    0.022996            2.30
6    market_KOSDAQ    0.015531            1.55
7     market_KONEX    0.003508            0.35
8     has_dividend    0.002667            0.27


,Feature,Importance,Percentage (%)
0,reduction_count,0.437363,43.74
1,log_profit,0.240183,24.02
2,dividend_rate,0.127275,12.73
3,par_price,0.091144,9.11
4,profit_negative,0.059333,5.93
5,market_KOSPI,0.022996,2.30
6,market_KOSDAQ,0.015531,1.55
7,market_KONEX,0.003508,0.35
8,has_dividend,0.002667,0.27


## 4. 딥러닝(PyTorch MLP) 모델 수치 평가
`models/risk_mlp.pt` 모델의 성능 지표를 산출하고, 머신러닝 모델과의 성능 비교 및 상관관계를 분석합니다.

In [9]:
dl_model, dl_preprocessor = load_dl_model()

# DL 전처리 수행
X_test_dl = dl_preprocessor.transform(X_test).astype(np.float32)

with torch.no_grad():
    logits = dl_model(torch.from_numpy(X_test_dl)).squeeze()
    y_prob_dl = torch.sigmoid(logits).numpy()

y_pred_dl = (y_prob_dl >= 0.5).astype(int)

# 지표 계산
acc_dl = float(accuracy_score(y_test, y_pred_dl))
prec_dl = float(precision_score(y_test, y_pred_dl, zero_division=0))
rec_dl = float(recall_score(y_test, y_pred_dl, zero_division=0))
f1_dl = float(f1_score(y_test, y_pred_dl, zero_division=0))
auc_dl = float(roc_auc_score(y_test, y_prob_dl))

bce_loss = float(torch.nn.functional.binary_cross_entropy(
    torch.from_numpy(y_prob_dl),
    torch.from_numpy(y_test.to_numpy().astype(np.float32))
).item())

print('=== 딥러닝(PyTorch MLP) 평가 결과 (테스트셋 N=548) ===')
print(f'Accuracy  : {acc_dl:.4f} ({acc_dl*100:.2f}%)')
print(f'Precision : {prec_dl:.4f}')
print(f'Recall    : {rec_dl:.4f}')
print(f'F1-Score  : {f1_dl:.4f}')
print(f'ROC-AUC   : {auc_dl:.4f}')
print(f'BCE Loss  : {bce_loss:.4f}')

# ML vs DL 비교 테이블
comparison_df = pd.DataFrame({
    '평가지표': ['정확도(Accuracy)', '정밀도(Precision)', '재현율(Recall)', 'F1-Score', 'ROC-AUC'],
    'ML (RandomForest)': [round(acc_ml, 4), round(prec_ml, 4), round(rec_ml, 4), round(f1_ml, 4), round(auc_ml, 4)],
    'DL (PyTorch MLP)': [round(acc_dl, 4), round(prec_dl, 4), round(rec_dl, 4), round(f1_dl, 4), round(auc_dl, 4)],
    '차이 (DL - ML)': [round(acc_dl - acc_ml, 4), round(prec_dl - prec_ml, 4), round(rec_dl - rec_ml, 4), round(f1_dl - f1_ml, 4), round(auc_dl - auc_ml, 4)]
})
print('\n=== ML vs DL 모델 성능 비교 ===')
comparison_df

=== 딥러닝(PyTorch MLP) 평가 결과 (테스트셋 N=548) ===
Accuracy  : 0.8175 (81.75%)
Precision : 0.2211
Recall    : 0.4468
F1-Score  : 0.2958
ROC-AUC   : 0.6909
BCE Loss  : 0.5195

=== ML vs DL 모델 성능 비교 ===
Out[0]: 
             평가지표  ML (RandomForest)  DL (PyTorch MLP)  차이 (DL - ML)
0   정확도(Accuracy)             0.8394            0.8175       -0.0219
1  정밀도(Precision)             0.2469            0.2211       -0.0259
2     재현율(Recall)             0.4255            0.4468        0.0213
3        F1-Score             0.3125            0.2958       -0.0167
4         ROC-AUC             0.6979            0.6909       -0.0070


,평가지표,ML (RandomForest),DL (PyTorch MLP),차이 (DL - ML)
0,정확도(Accuracy),0.8394,0.8175,-0.0219
1,정밀도(Precision),0.2469,0.2211,-0.0259
2,재현율(Recall),0.4255,0.4468,0.0213
3,F1-Score,0.3125,0.2958,-0.0167
4,ROC-AUC,0.6979,0.6909,-0.0070


In [10]:
# 전체 2,737개사에 대한 ML 및 DL 예측 점수(0~100) 산출
all_ml_probs = ml_model.predict_proba(X)[:, 1] * 100
all_X_dl = dl_preprocessor.transform(X).astype(np.float32)

with torch.no_grad():
    all_dl_probs = torch.sigmoid(dl_model(torch.from_numpy(all_X_dl)).squeeze()).numpy() * 100

score_diff = np.abs(all_ml_probs - all_dl_probs)
score_corr = float(np.corrcoef(all_ml_probs, all_dl_probs)[0, 1])

diff_stats = pd.Series(score_diff).describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).round(2)
print('=== ML vs DL 점수 일치도 및 상관관계 분석 (전체 2,737개사) ===')
print(f'피어슨 상관계수 (ML 점수 vs DL 점수): {score_corr:.4f}')
print(f'평균 절대 오차 (MAE): {np.mean(score_diff):.2f}점')
print('\n점수 차이(절댓값) 분포 통계:')
print(diff_stats)

=== ML vs DL 점수 일치도 및 상관관계 분석 (전체 2,737개사) ===
피어슨 상관계수 (ML 점수 vs DL 점수): 0.8746
평균 절대 오차 (MAE): 6.98점

점수 차이(절댓값) 분포 통계:
count    2737.00
mean        6.98
std         5.96
min         0.00
25%         2.76
50%         5.73
75%         9.48
90%        13.91
95%        17.62
max        46.78
dtype: float64


## 5. 앙상블 종합 점수 및 등급 규칙 수치 검증
ML 점수와 DL 점수의 산술 평균으로 최종 참고 위험 점수를 계산하고, 등급 판정 규칙(`stable`, `warning`, `danger`)을 적용한 전수 분포를 분석합니다.

In [11]:
final_scores = np.round((all_ml_probs + all_dl_probs) / 2, 1)

# 전수 등급 계산
assigned_grades = []
danger_due_to_disclosure = 0
warning_due_to_score = 0
warning_due_to_reduction = 0

for i, row in df.iterrows():
    s = final_scores[i]
    g = grade_of(s, row)
    assigned_grades.append(g)
    
    if g == 'danger':
        if row['default_count'] > 0 or row['suspend_count'] > 0 or row['rehab_count'] > 0:
            danger_due_to_disclosure += 1
    elif g == 'warning':
        if s >= SCORE_THRESHOLD_WARNING:
            warning_due_to_score += 1
        elif row['reduction_count'] > 0:
            warning_due_to_reduction += 1

grade_counts = pd.Series(assigned_grades).value_counts()
grade_summary = pd.DataFrame({
    '등급': ['안정 (stable)', '주의 (warning)', '위험 (danger)'],
    '기업 수': [int(grade_counts.get('stable', 0)), int(grade_counts.get('warning', 0)), int(grade_counts.get('danger', 0))],
    '비율 (%)': [
        round(grade_counts.get('stable', 0) / len(df) * 100, 2),
        round(grade_counts.get('warning', 0) / len(df) * 100, 2),
        round(grade_counts.get('danger', 0) / len(df) * 100, 2)
    ],
    '설명': [
        '위험점수 40점 미만 & 부도·영업정지·회생·감자 없음',
        '점수 40점 이상(또는 감자 공시 이력)',
        '부도·영업정지·회생 공시 발생 (사실 우선 원칙)'
    ]
})

print('=== 전수 위험 등급 판정 분포 (N=2,737) ===')
print(grade_summary)

print('\n=== 규칙 적용 세부 분석 ===')
print(f'- 부도·영업정지·회생 공시로 위험(danger) 등급 승격된 기업: {danger_due_to_disclosure}개사')
print(f'- 모델 점수 40점 이상으로 주의(warning) 등급 부여된 기업: {warning_due_to_score}개사')
print(f'- 감자 공시 이력으로 주의(warning) 등급 부여된 기업: {warning_due_to_reduction}개사')

=== 전수 위험 등급 판정 분포 (N=2,737) ===
             등급  기업 수  비율 (%)                              설명
0   안정 (stable)  1722   62.92  위험점수 40점 미만 & 부도·영업정지·회생·감자 없음
1  주의 (warning)   782   28.57          점수 40점 이상(또는 감자 공시 이력)
2   위험 (danger)   233    8.51     부도·영업정지·회생 공시 발생 (사실 우선 원칙)

=== 규칙 적용 세부 분석 ===
- 부도·영업정지·회생 공시로 위험(danger) 등급 승격된 기업: 233개사
- 모델 점수 40점 이상으로 주의(warning) 등급 부여된 기업: 763개사
- 감자 공시 이력으로 주의(warning) 등급 부여된 기업: 19개사


## 6. 기능 2 - 출자금 원 단위 배분(Allocation Split) 수치 계산 검증
Spring 백엔드의 `AllocationCalculator.java`와 동일한 원 단위 분할 및 절사 잔여액 배분 알고리즘을 구현하고, 다양한 출자 시나리오에서 오차 0원 검증을 수행합니다.

In [12]:
def split_allocation(total_amount: int, ratios: list[float]) -> list[int]:
    '''
    백엔드 AllocationCalculator.split과 동일한 로직:
    각 출자자 배분액 = floor(total_amount * ratio / 100)
    마지막 출자자가 잔여 원을 모두 가져가 합계 = 투자금 보장
    '''
    amounts = []
    used = 0
    for i, ratio in enumerate(ratios):
        is_last = (i == len(ratios) - 1)
        if is_last:
            amounts.append(total_amount - used)
        else:
            part = int(Decimal(str(total_amount)) * Decimal(str(ratio)) / Decimal('100'))
            amounts.append(part)
            used += part
    return amounts

# 시나리오 1: 1억원 3명 균등 (33.33%, 33.33%, 33.34%)
test_1_total = 100_000_000
test_1_ratios = [33.33, 33.33, 33.34]
res_1 = split_allocation(test_1_total, test_1_ratios)

# 시나리오 2: 10억원 3명 차등 (50.0%, 30.0%, 20.0%)
test_2_total = 1_000_000_000
test_2_ratios = [50.0, 30.0, 20.0]
res_2 = split_allocation(test_2_total, test_2_ratios)

# 시나리오 3: 소수점 오차가 발생하는 비정형 금액 (77,777,777원)
test_3_total = 77_777_777
test_3_ratios = [14.28, 28.57, 57.15]
res_3 = split_allocation(test_3_total, test_3_ratios)

alloc_df = pd.DataFrame([
    {'시나리오': '1. 1억원 균등 배분', '총투자금': f'{test_1_total:,}원', '비율 목록': str(test_1_ratios), '배분액 결과': [f'{x:,}원' for x in res_1], '합계 일치': sum(res_1) == test_1_total},
    {'시나리오': '2. 10억원 차등 배분', '총투자금': f'{test_2_total:,}원', '비율 목록': str(test_2_ratios), '배분액 결과': [f'{x:,}원' for x in res_2], '합계 일치': sum(res_2) == test_2_total},
    {'시나리오': '3. 77,777,777원 난수 배분', '총투자금': f'{test_3_total:,}원', '비율 목록': str(test_3_ratios), '배분액 결과': [f'{x:,}원' for x in res_3], '합계 일치': sum(res_3) == test_3_total},
])

print('=== 출자금 배분 알고리즘 수치 검증 ===')
alloc_df

=== 출자금 배분 알고리즘 수치 검증 ===
Out[0]: 
                   시나리오  ... 합계 일치
0          1. 1억원 균등 배분  ...  True
1         2. 10억원 차등 배분  ...  True
2  3. 77,777,777원 난수 배분  ...  True

[3 rows x 5 columns]


,시나리오,총투자금,비율 목록,배분액 결과,합계 일치
0,1. 1억원 균등 배분,"100,000,000원","[33.33, 33.33, 33.34]","[33,330,000원, 33,330,000원, 33,340,000원]",True
1,2. 10억원 차등 배분,"1,000,000,000원","[50.0, 30.0, 20.0]","[500,000,000원, 300,000,000원, 200,000,000원]",True
2,"3. 77,777,777원 난수 배분","77,777,777원","[14.28, 28.57, 57.15]","[11,106,666원, 22,221,110원, 44,450,001원]",True


## 7. 실제 상장 종목 E2E 진단 케이스 스터디
모듈화된 AI 파이프라인(`src.service.screener.screen`)을 통해 실제 주요 기업들의 진단 수치 및 규칙 적용 결과를 확인합니다.

In [13]:
test_target_companies = ['삼성전자', '카카오']

# 감자 이력만 있는 기업(부도/정지/회생 없음) 1곳 선정 -> '주의' 등급 규칙 검증
red_only = df[(df['reduction_count'] > 0) & (df['default_count'] == 0) & (df['suspend_count'] == 0) & (df['rehab_count'] == 0)]['item_name'].tolist()
if red_only:
    test_target_companies.append(red_only[0])

# 부도/영업정지/회생 이력이 있는 고위험 기업 1곳 선정 -> '위험' 등급 자동 승격 검증
danger_cos = df[(df['default_count'] > 0) | (df['suspend_count'] > 0) | (df['rehab_count'] > 0)]['item_name'].tolist()
if danger_cos:
    test_target_companies.append(danger_cos[0])

case_results = []
for cname in test_target_companies:
    res = screen(cname)
    case_results.append({
        '종목명': res['company_name'],
        '단축코드': res['stock_code'],
        '시장': res['market'],
        'ML점수': res['ml_score'],
        'DL점수': res['dl_score'],
        '종합점수': res['score'],
        '판정등급': GRADE_NAMES.get(res['grade'], res['grade']),
        '배당률': f"{res['dividend_rate']}%",
        '순이익(억원)': f"{res['net_profit'] / 1e8:,.1f}",
        '부도/정지/회생': f"{res['default_count']}/{res['suspend_count']}/{res['rehab_count']}",
        '감자건수': res['reduction_count'],
        '설명 출처': res['explanation_source']
    })

case_df = pd.DataFrame(case_results)
print('=== 실제 상장 기업 E2E 진단 결과 요약 ===')
case_df

=== 실제 상장 기업 E2E 진단 결과 요약 ===
Out[0]: 
      종목명     단축코드     시장  ML점수  DL점수  ...   배당률 순이익(억원) 부도/정지/회생 감자건수  설명 출처
0    삼성전자  A005930  KOSPI  24.0  11.2  ...  2.7%     0.1    0/0/0    0  basic
1     카카오  A035720  KOSPI  71.7  90.1  ...  0.2%    -0.0    0/0/0    5  basic
2  LS네트웍스  A000680  KOSPI  58.8  34.3  ...  0.0%     0.0    0/0/0    1  basic
3   KR모터스  A000040  KOSPI  88.0  86.0  ...  0.0%    -0.0    0/1/0    3  basic

[4 rows x 12 columns]


,종목명,단축코드,시장,ML점수,DL점수,종합점수,판정등급,배당률,순이익(억원),부도/정지/회생,감자건수,설명 출처
0,삼성전자,A005930,KOSPI,24.0,11.2,17.6,안정,2.7%,0.1,0/0/0,0,basic
1,카카오,A035720,KOSPI,71.7,90.1,80.9,주의,0.2%,-0.0,0/0/0,5,basic
2,LS네트웍스,A000680,KOSPI,58.8,34.3,46.5,주의,0.0%,0.0,0/0/0,1,basic
3,KR모터스,A000040,KOSPI,88.0,86.0,87.0,위험,0.0%,-0.0,0/1/0,3,basic


In [14]:
print('=== 대표 진단 설명 출력 예시 (삼성전자) ===')
sample_screen = screen('삼성전자')
print(sample_screen['explanation'])
print(f"\n[출처: {sample_screen['explanation_source']}]")

=== 대표 진단 설명 출력 예시 (삼성전자) ===
삼성전자(KOSPI)의 참고 위험 점수는 17.6점으로 '안정' 단계입니다. 머신러닝 24.0점, 딥러닝 11.2점입니다. 수집한 공시에서는 부도·영업정지·회생·감자가 보이지 않습니다. 이 점수는 출자 여부를 대신 정해 주지 않으며, 공시와 상장 정보를 요약한 참고값입니다.

[출처: basic]
